In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.pipeline import  Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder,RobustScaler,TargetEncoder
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.ensemble import  RandomForestRegressor
from sklearn.ensemble import BaggingRegressor
from  sklearn.metrics import r2_score
import seaborn as sns
from sklearn.svm import  SVR

In [13]:
df=pd.read_csv('Data_Train.csv')

In [14]:
df.head(1)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR ? DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 918.2+ KB


In [16]:
df=df.dropna()

In [17]:
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR ? DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL ? LKO ? BOM ? COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU ? NAG ? BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR ? NAG ? DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [18]:
df['Airline'].unique()

array(['IndiGo', 'Air India', 'Jet Airways', 'SpiceJet',
       'Multiple carriers', 'GoAir', 'Vistara', 'Air Asia',
       'Vistara Premium economy', 'Jet Airways Business',
       'Multiple carriers Premium economy', 'Trujet'], dtype=object)

In [19]:
df['Source'].unique()

array(['Banglore', 'Kolkata', 'Delhi', 'Chennai', 'Mumbai'], dtype=object)

In [20]:
df['Destination'].unique()

array(['New Delhi', 'Banglore', 'Cochin', 'Kolkata', 'Delhi', 'Hyderabad'],
      dtype=object)

In [21]:
df['Route'].unique().shape

(128,)

In [22]:
df['Additional_Info'].unique()

array(['No info', 'In-flight meal not included',
       'No check-in baggage included', '1 Short layover', 'No Info',
       '1 Long layover', 'Change airports', 'Business class',
       'Red-eye flight', '2 Long layover'], dtype=object)

In [23]:
df=df.drop(columns=['Additional_Info','Route'])

In [24]:
def duration(row):
    hr=0
    min=0
    if 'h' in row:
        hr=int(row.split('h')[0])
    if 'm' in row:
        min=int(row.split('m')[0].split()[-1])
    return hr*60 + min

In [25]:
df['Duration_min']=df['Duration'].apply(duration)
df.drop('Duration',axis=1,inplace=True)

In [26]:
df['Total_Stops'].unique()

array(['non-stop', '2 stops', '1 stop', '3 stops', '4 stops'],
      dtype=object)

In [27]:
df['Total_Stops']=df['Total_Stops'].replace({'non-stop':0,'2 stops':2,'1 stop':1,'3 stops':3,'4 stops':4})

C:\Users\Admin\AppData\Local\Temp\ipykernel_8804\252118392.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Total_Stops']=df['Total_Stops'].replace({'non-stop':0,'2 stops':2,'1 stop':1,'3 stops':3,'4 stops':4})


In [28]:
df.head()

,Airline,Date_of_Journey,Source,Destination,Dep_Time,Arrival_Time,Total_Stops,Price,Duration_min
0,IndiGo,24/03/2019,Banglore,New Delhi,22:20,01:10 22 Mar,0,3897,170
1,Air India,1/05/2019,Kolkata,Banglore,05:50,13:15,2,7662,445
2,Jet Airways,9/06/2019,Delhi,Cochin,09:25,04:25 10 Jun,2,13882,1140
3,IndiGo,12/05/2019,Kolkata,Banglore,18:05,23:30,1,6218,325
4,IndiGo,01/03/2019,Banglore,New Delhi,16:50,21:35,1,13302,285


In [29]:
df['Date_of_Journey']=pd.to_datetime(df['Date_of_Journey'])
df['date_month']=df['Date_of_Journey'].dt.month
df['date_day']=df['Date_of_Journey'].dt.day
df.drop("Date_of_Journey",axis=1,inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_8804\453012238.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date_of_Journey']=pd.to_datetime(df['Date_of_Journey'])


In [30]:
df['Dep_Time']=pd.to_datetime(df['Dep_Time'])
df['dep_hour']=df['Dep_Time'].dt.hour
df['dep_min']=df['Dep_Time'].dt.minute
df.drop("Dep_Time",axis=1,inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_8804\2499187102.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Dep_Time']=pd.to_datetime(df['Dep_Time'])


In [31]:
df['Arrival_Time']=pd.to_datetime(df['Arrival_Time'])
df['Arrival_hour']=df['Arrival_Time'].dt.hour
df['Arrival_min']=df['Arrival_Time'].dt.minute
df.drop("Arrival_Time",axis=1,inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_8804\2376069642.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Arrival_Time']=pd.to_datetime(df['Arrival_Time'])


In [32]:
df.head(3)

,Airline,Source,Destination,Total_Stops,Price,Duration_min,date_month,date_day,dep_hour,dep_min,Arrival_hour,Arrival_min
0,IndiGo,Banglore,New Delhi,0,3897,170,3,24,22,20,1,10
1,Air India,Kolkata,Banglore,2,7662,445,5,1,5,50,13,15
2,Jet Airways,Delhi,Cochin,2,13882,1140,6,9,9,25,4,25


In [33]:
x=df.drop('Price',axis=1)
y=df.Price

In [34]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [35]:
num_col=xtrain.select_dtypes(include='number').columns
obj_col=xtrain.select_dtypes(exclude='number').columns

In [36]:
preprocessing=ColumnTransformer(
    transformers=[
        ('scaler',RobustScaler(),num_col),
        ('target_encoder',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),obj_col)
    ]
)

In [37]:
decisiontree_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model',DecisionTreeRegressor())
    ]
)

In [38]:
decisiontree_pipeline.fit(xtrain,ytrain)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('target_encoder', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [39]:
print("Train_score:",decisiontree_pipeline.score(xtrain,ytrain))
print("Test_score:",decisiontree_pipeline.score(xtest,ytest))    

Train_score: 0.9692484150527356
Test_score: 0.6808764377882395


In [40]:
# grid_search_cv=GridSearchCV(
#     estimator=Pipeline,
#     param_grid={'max_depth':[None,5,10,15,20],
#     'min_samples_split':[2,4,6,8,10],
#     'min_samples_leaf':[1,3,5,7,10],
#     'Criterion':['absolute_error','squared_error']
#     },
#     cv=3,
#     n_jobs=-1,
#     verbose=1,
#     scoring='neg_mean_absolute_error'
# )

In [41]:
# grid_search_cv.fit(xtrain,ytrain)

In [42]:
# Train_score: 0.7891466947957774
# Test_score: 0.6569504686785084
# Train_score: 0.6622215082027301
# Test_score: 0.6713781764208201

In [43]:
Random_forest_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model',RandomForestRegressor(max_depth=15,min_samples_split=10,min_samples_leaf=5,criterion='absolute_error',max_features='sqrt',n_estimators=200))
    ]
)

In [44]:
Random_forest_pipeline.fit(xtrain,ytrain)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('target_encoder', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [45]:
print("train_score:",Random_forest_pipeline.score(xtrain,ytrain))
print("test_score:",Random_forest_pipeline.score(xtest,ytest))

train_score: 0.8012455760669208
test_score: 0.7853440053892606
